Connected to .venv (Python 3.13.2)

In [ ]:
import pandas as pd

In [ ]:
# Load data (keep all columns)
df = pd.read_csv('zt_logs.csv')

In [ ]:
# Ensure timestamp exists: create from date+time if necessary, otherwise parse existing timestamp
if 'timestamp' not in df.columns and {'date', 'time'}.issubset(df.columns):
    df['timestamp'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['time'].astype(str), errors='coerce')
else:
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

In [ ]:
# Sort by user then timestamp (keeps all columns)
df = df.sort_values(by=['user', 'timestamp'], ascending=[True, True]).reset_index(drop=True)

In [ ]:
# Compute time diff within each user (seconds and minutes)
df['time_diff_seconds'] = df.groupby('user')['timestamp'].diff().dt.total_seconds()
df['time_diff_minutes'] = df['time_diff_seconds'] / 60

In [ ]:
# Get previous country for same user to detect change
df['prev_country'] = df.groupby('user')['country'].shift(1)

In [ ]:
# Anomaly rule: time_diff_minutes < 180 AND country changed (prev_country != country)
df['is_anomaly'] = (
    df['time_diff_minutes'].notna()  # exclude first events which are NaN
    & (df['time_diff_minutes'] < 180)
    & (df['country'] != df['prev_country'])
).astype(int)

In [ ]:
df.head(15)

,timestamp,user,role,device_id,device_trust,country,lat,lon,resource,action,result,mfa,is_anomaly,time_diff_seconds,time_diff_minutes,prev_country
0,2025-10-10 03:35:18,user001,contractor,device022,untrusted,US,37.77,-122.41,email,login,success,False,0,NaN,NaN,NaN
1,2025-10-10 12:58:46,user001,employee,device071,trusted,UK,51.50,-0.12,vpn,access,success,True,0,33808.0,563.466667,US
2,2025-10-10 12:58:49,user001,employee,device016,trusted,UK,51.50,-0.12,hr_portal,logout,success,True,0,3.0,0.050000,UK
3,2025-10-10 12:59:01,user001,employee,device074,trusted,SG,1.35,103.82,finance_app,access,success,False,1,12.0,0.200000,UK
4,2025-10-10 13:12:38,user001,employee,device007,trusted,US,37.77,-122.41,email,access,success,True,1,817.0,13.616667,SG
5,2025-10-10 13:24:32,user001,contractor,device025,trusted,UK,51.50,-0.12,vpn,access,success,False,1,714.0,11.900000,US
6,2025-10-10 13:25:08,user001,employee,device051,untrusted,IN,19.07,72.87,vpn,login,success,True,1,36.0,0.600000,UK
7,2025-10-10 13:26:07,user001,employee,device046,trusted,IN,19.07,72.87,finance_app,logout,success,False,0,59.0,0.983333,IN
8,2025-10-10 13:37:20,user001,employee,device017,trusted,DE,52.52,13.40,email,logout,success,True,1,673.0,11.216667,IN
9,2025-10-10 13:50:59,user001,admin,device030,trusted,SG,1.35,103.82,dev_repo,access,success,False,1,819.0,13.650000,DE
